# Download the 10 newest arXiv papers matching a keyword

Edit **`KEYWORD`** in the configuration cell and run **Run All**. This notebook searches the official arXiv API, sorts by **original submission date (newest first)**, downloads the matching PDFs, and saves paper metadata as CSV and JSON. It skips existing valid PDFs and records download failures.

**Meaning of “latest”:** latest initial arXiv submissions, *not* latest revisions or publication dates. Change `SORT_BY` to `lastUpdatedDate` to prioritize recently revised papers. `SEARCH_FIELD="all"` searches arXiv's indexed metadata, **not every sentence in the PDF**.

The API may return fewer than ten results for a narrow query. An API request retrieves at most `MAX_PAPERS` metadata records; if a PDF download fails, it is reported rather than silently replaced by an older paper.

In [ ]:
%pip install -q requests pandas

## 1. Configuration

Set a keyword such as `quantum graph neural network`, `jet tagging`, or `Higgs boson`. Multiword keywords are searched as an **exact phrase** by default; set `EXACT_PHRASE = False` to match all words independently (AND). For advanced arXiv queries, enter the complete expression in `CUSTOM_QUERY` (e.g., `cat:hep-ph AND all:"machine learning"`).

In [ ]:
from pathlib import Path

KEYWORD = "quantum machine learning"   # <-- CHANGE THIS
MAX_PAPERS = 10
SEARCH_FIELD = "all"                # "all", "ti" (title), or "abs" (abstract)
EXACT_PHRASE = True
CUSTOM_QUERY = None                  # e.g. 'cat:hep-ph AND all:"machine learning"'
SORT_BY = "submittedDate"            # alternatively "lastUpdatedDate"
OUTPUT_DIR = Path("arxiv_papers")
REQUEST_DELAY_SECONDS = 3           # polite gap between arXiv downloads

assert MAX_PAPERS > 0
assert SEARCH_FIELD in {"all", "ti", "abs"}
assert SORT_BY in {"submittedDate", "lastUpdatedDate"}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())

## 2. Search the arXiv API and parse the Atom response

HTTP retries help with temporary failures. An HTTP error or malformed XML raises a visible exception rather than creating an empty, misleading dataset.

In [ ]:
import json
import re
import time
import xml.etree.ElementTree as ET
from urllib.parse import urlparse

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_URL = "https://export.arxiv.org/api/query"
ATOM = {"a": "http://www.w3.org/2005/Atom"}

session = requests.Session()
session.headers.update({
    "User-Agent": "PhysicsPaperDownloader/1.0 (academic research; Python requests)"
})
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"], respect_retry_after_header=True
)))


def build_query(keyword, field="all", exact_phrase=True, custom_query=None):
    if custom_query:
        return custom_query.strip()
    words = keyword.strip().split()
    if not words:
        raise ValueError("KEYWORD must not be empty.")
    # Avoid placing embedded quotes into arXiv's phrase syntax.
    words = [w.replace('"', '') for w in words]
    if exact_phrase:
        return f'{field}:"{" ".join(words)}"'
    return " AND ".join(f"{field}:{word}" for word in words)


def parse_entry(entry):
    def text_of(path):
        item = entry.find(path, ATOM)
        return " ".join(item.text.split()) if item is not None and item.text else ""

    abstract_url = text_of("a:id")
    # API ids can include a version, which we preserve in metadata.
    arxiv_id = urlparse(abstract_url).path.removeprefix("/abs/")
    if not re.fullmatch(r"(?:\d{4}\.\d{4,5}|[a-z-]+(?:\.[A-Z]{2})?/\d{7})(?:v\d+)?", arxiv_id, re.I):
        raise ValueError(f"Unexpected arXiv identifier: {arxiv_id!r}")

    authors = [" ".join(a.findtext("a:name", default="", namespaces=ATOM).split())
               for a in entry.findall("a:author", ATOM)]
    pdf_link = next((link.attrib["href"] for link in entry.findall("a:link", ATOM)
                     if link.attrib.get("title") == "pdf" or link.attrib.get("type") == "application/pdf"),
                    f"https://arxiv.org/pdf/{arxiv_id}")
    return {
        "arxiv_id": arxiv_id,
        "title": text_of("a:title"),
        "authors": ", ".join(a for a in authors if a),
        "published": text_of("a:published"),
        "updated": text_of("a:updated"),
        "abstract": text_of("a:summary"),
        "categories": ", ".join(x.attrib.get("term", "") for x in entry.findall("a:category", ATOM)),
        "abstract_url": abstract_url,
        "pdf_url": pdf_link,
    }


def search_arxiv(query, maximum=10, sort_by="submittedDate"):
    params = {
        "search_query": query, "start": 0, "max_results": maximum,
        "sortBy": sort_by, "sortOrder": "descending",
    }
    response = session.get(API_URL, params=params, timeout=(15, 90))
    response.raise_for_status()
    try:
        root = ET.fromstring(response.content)
    except ET.ParseError as exc:
        raise RuntimeError("arXiv returned something other than valid Atom XML") from exc
    if root.tag != "{http://www.w3.org/2005/Atom}feed":
        raise RuntimeError("Unexpected arXiv API response (not an Atom feed).")
    return [parse_entry(e) for e in root.findall("a:entry", ATOM)]

query = build_query(KEYWORD, SEARCH_FIELD, EXACT_PHRASE, CUSTOM_QUERY)
print("Search query:", query)
papers = search_arxiv(query, MAX_PAPERS, SORT_BY)
print(f"Found {len(papers)} papers (requested {MAX_PAPERS}).")
if papers:
    display(pd.DataFrame(papers)[["arxiv_id", "published", "title", "authors"]])
else:
    print("No matches. Try EXACT_PHRASE=False, SEARCH_FIELD='all', or a broader keyword.")

## 3. Download PDFs

Files are named by arXiv ID, which prevents collisions between papers with identical titles. A temporary `.part` file is only renamed after a PDF signature check; an existing PDF is left untouched.

In [ ]:
def download_pdf(paper, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        with destination.open("rb") as f:
            if f.read(5) == b"%PDF-":
                return "already_exists"
        print(f"Replacing invalid existing PDF: {destination.name}")

    # Keep downloads on arxiv.org rather than following an arbitrary PDF URL.
    paper_id = paper["arxiv_id"]
    pdf_url = f"https://arxiv.org/pdf/{paper_id}"
    partial = destination.with_suffix(".pdf.part")
    partial.unlink(missing_ok=True)
    try:
        with session.get(pdf_url, stream=True, timeout=(15, 120)) as response:
            response.raise_for_status()
            with partial.open("wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 256):
                    if chunk:
                        f.write(chunk)
        with partial.open("rb") as f:
            if f.read(5) != b"%PDF-":
                raise ValueError("Response was not a PDF (missing %PDF- signature)")
        partial.replace(destination)
        return "downloaded"
    except Exception:
        partial.unlink(missing_ok=True)
        raise


results = []
if papers:
    time.sleep(REQUEST_DELAY_SECONDS)  # pause after the API query before the first PDF request
for index, paper in enumerate(papers, start=1):
    # ArXiv identifier is validated in parse_entry; slash is only relevant to older ids.
    safe_id = paper["arxiv_id"].replace("/", "_")
    destination = OUTPUT_DIR / f"{safe_id}.pdf"
    row = {**paper, "pdf_file": str(destination), "download_status": "", "error": ""}
    try:
        row["download_status"] = download_pdf(paper, destination)
        print(f"[{index}/{len(papers)}] {row['download_status']}: {paper['title'][:90]}")
    except (requests.RequestException, OSError, ValueError) as exc:
        row["download_status"] = "failed"
        row["error"] = f"{type(exc).__name__}: {exc}"
        print(f"[{index}/{len(papers)}] FAILED: {paper['arxiv_id']}: {row['error']}")
    results.append(row)
    if index < len(papers):
        time.sleep(REQUEST_DELAY_SECONDS)

metadata = pd.DataFrame(results)
metadata.to_csv(OUTPUT_DIR / "metadata.csv", index=False, encoding="utf-8")
(OUTPUT_DIR / "metadata.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

success_count = sum(row["download_status"] in {"downloaded", "already_exists"} for row in results)
print(f"\nAvailable PDFs: {success_count}/{len(papers)}")
print("Saved metadata:", OUTPUT_DIR / "metadata.csv", "and", OUTPUT_DIR / "metadata.json")
print("PDF folder:", OUTPUT_DIR.resolve())
if len(metadata):
    display(metadata[["arxiv_id", "title", "download_status", "pdf_file", "error"]])

## Notes

- This is a metadata search, not full-PDF text search. For subject-specific searches you can use `CUSTOM_QUERY = 'cat:hep-ph AND all:"machine learning"'`.
- If the same papers appear on a second run, existing PDF files will not be downloaded again. arXiv may update papers, so a previously saved version remains untouched unless you delete it.
- The notebook saves PDFs and metadata to `arxiv_papers/` relative to your notebook's working directory. Use that directory as input for your subsequent PDF extraction or QLoRA workflow.
- arXiv asks API clients to leave around three seconds between requests. This notebook spaces PDF requests and uses HTTP retry backoff. For bulk datasets, consider arXiv's bulk access rather than repeatedly querying the API.

Documentation: [arXiv API user manual](https://info.arxiv.org/help/api/user-manual.html).